# 🚀 AGAR-RL V2 : Pipeline Autonome Deep RL (GPU L4 - 15M Steps)

**Entraînement de haute performance à pleine puissance (2h à 3h max sur GPU NVIDIA L4)**.
Ce notebook est entièrement conçu pour le mode **« Exécuter tout » (Run All)** :
1. Connexion & Sauvegarde automatique en temps réel sur **Google Drive** (`agario_rl_backup_v2`).
2. Validation immédiate des 27 tests unitaires du moteur physique.
3. Entraînement de 15 000 000 de pas avec la nouvelle formulation mathématique (Reward log, Anti-Camping, Anti-Suicide Split, Masquage d'actions).
4. Enregistrement automatique de la vidéo HD du match final et export du modèle **ONNX**.

## 0. Montage Google Drive & Détection GPU L4

In [ ]:
from google.colab import drive
import os, sys, time, torch

# 1. Montage du Google Drive
drive.mount('/content/drive')
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/agario_rl_backup_v2'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# 2. Test d'écriture
test_file = os.path.join(DRIVE_BACKUP_DIR, 'test_v2.txt')
with open(test_file, 'w', encoding='utf-8') as f:
    f.write(f'AGAR-RL V2 Google Drive OK - {time.ctime()}\n')

print(f'✅ Google Drive connecté. Sauvegardes dirigées vers : {DRIVE_BACKUP_DIR}')
print(f'🚀 CUDA Disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'⚡ GPU Actif : {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ ATTENTION : Vous êtes sur CPU ! Allez dans Exécution > Modifier le type d\'exécution > GPU L4.')


## 1. Synchronisation du Code GitHub & Installation des Dépendances

In [ ]:
import os

# 1. Récupération des dernières modifications ou clonage
if os.path.exists('.git'):
    print('🔄 Récupération des dernières mises à jour du repo...')
    !git pull origin main
elif os.path.exists('agario/.git'):
    print('🔄 Déplacement dans agario et mise à jour...')
    %cd agario
    !git pull origin main
else:
    print('🌐 Clonage propre du repo...')
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

# 2. Configuration du PYTHONPATH et installation des dépendances Farama Gymnasium
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip uninstall -y -q gym 2>/dev/null || true
!pip install -q -r requirements.txt tensorboard
!apt-get install -qq -y ffmpeg
print('✅ Environnement et dépendances installés.')


## 2. Validation Pré-Vol : Suite de 27 Tests Unitaires

In [ ]:
# Exécute tous les tests du moteur physique Ogar, des règles Gymnasium et de l'export ONNX
!python -m pytest -v


## 3. Monitoring TensorBoard (Optionnel)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/tensorboard


## 4. Entraînement Haute Performance V2 (15 000 000 Steps ≈ 2h15 sur GPU L4)

**Caractéristiques du run V2** :
* **16 environnements parallèles** (`SubprocVecEnv`) générant ~1 800 FPS.
* **Horizon de rollout de 2 048 pas** par worker (batch de 32 768 pas, mini-batch de 512).
* **Réseau Actor-Critic 512x512** pour une représentation spatiale fine.
* **Croissance logarithmique relative** ($R_{\text{growth}} = \ln(M_t / M_{t-1})$).
* **Anti-Corner Camping** (répulsion quadratique des murs de $D_{\text{safe}} = 120\text{ px}$).
* **Anti-Suicide Split (-5.0)** et **Pénalité de catastrophe virus (-30.0)**.
* **Éjection solo désactivée** (aucun gaspillage de masse).
* **Ligue de Self-Play** : 40% clones récents, 30% archives figées, 30% heuristiques.

In [ ]:
# 🚀 Lancement à pleine puissance (15 000 000 timesteps ≈ 2h15 à 1800 FPS sur GPU L4)
# Sauvegarde miroir automatique sur Google Drive toutes les 250 000 étapes
!python src/training/train_colab.py \
    --n-envs 16 \
    --total-timesteps 15000000 \
    --pool-interval 250000 \
    --backup-dir /content/drive/MyDrive/agario_rl_backup_v2 \
    --resume none \
    --device auto


## 5. Enregistrement Automatique du Match Replay HD & Visualisation Directe

In [ ]:
import os, glob, re
from IPython.display import HTML, display
from base64 import b64encode

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

# 1. Recherche automatique du modèle final 15M (Drive ou local)
candidates = [
    'checkpoints/ppo/ppo_final.zip',
    '/content/drive/MyDrive/agario_rl_backup_v2/ppo_final.zip',
]
target_model = next((c for c in candidates if os.path.exists(c)), None)

if not target_model:
    all_ckpts = glob.glob('/content/drive/MyDrive/agario_rl_backup_v2/*.zip') + glob.glob('checkpoints/self_play_pool/*.zip')
    if all_ckpts:
        all_ckpts.sort(key=extract_step)
        target_model = all_ckpts[-1]
    else:
        target_model = 'checkpoints/ppo/ppo_latest.zip'

step_count = extract_step(target_model)
print('=' * 75)
print(f'🎬 Modèle sélectionné pour l\'évaluation HD : {target_model}')
print(f'📊 Palier de pas : {step_count:,} steps')
print('=' * 75)

# 2. Enregistrement du replay (2400 steps @ 30 FPS = 80 secondes de vidéo HD)
os.makedirs('recordings', exist_ok=True)
!python src/inference/record_match.py \
    --model "{target_model}" \
    --output recordings/eval_match_v2.mp4 \
    --steps 2400

# 3. Sauvegarde sur Google Drive
if os.path.exists('recordings/eval_match_v2.mp4') and os.path.exists('/content/drive/MyDrive/agario_rl_backup_v2'):
    !cp recordings/eval_match_v2.mp4 /content/drive/MyDrive/agario_rl_backup_v2/eval_match_v2.mp4
    print('📁 Replay HD copié sur Google Drive dans : agario_rl_backup_v2/eval_match_v2.mp4')

# 4. Visualisation directe dans le Notebook
video_path = 'recordings/eval_match_v2.mp4'
if os.path.exists(video_path):
    mp4_bytes = open(video_path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <video width="850" height="480" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
    print(f'Taille de la vidéo : {os.path.getsize(video_path) / 1_000_000:.1f} Mo')
else:
    print('⚠️ Vidéo non trouvée.')


## 6. Exportation Universelle vers ONNX & Benchmark de Latence

In [ ]:
import os, glob, re

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

candidates = [
    'checkpoints/ppo/ppo_final.zip',
    '/content/drive/MyDrive/agario_rl_backup_v2/ppo_final.zip',
]
target_model = next((c for c in candidates if os.path.exists(c)), None)
if not target_model:
    all_ckpts = glob.glob('/content/drive/MyDrive/agario_rl_backup_v2/*.zip') + glob.glob('checkpoints/self_play_pool/*.zip')
    all_ckpts.sort(key=extract_step)
    target_model = all_ckpts[-1] if all_ckpts else 'checkpoints/ppo/ppo_latest.zip'

os.makedirs('models', exist_ok=True)
!python src/inference/export_onnx.py \
    --model "{target_model}" \
    --output models/model_v2.onnx

if os.path.exists('models/model_v2.onnx') and os.path.exists('/content/drive/MyDrive/agario_rl_backup_v2'):
    !cp models/model_v2.onnx /content/drive/MyDrive/agario_rl_backup_v2/model_v2.onnx
    print('📁 Modèle ONNX sauvegardé sur Google Drive dans : agario_rl_backup_v2/model_v2.onnx')

print('\n' + '=' * 75)
print('🎉 PIPELINE AGAR-RL V2 TERMINÉ AVEC SUCCÈS !')
print('Toutes vos données (checkpoints, replay HD, modèle ONNX) sont sécurisées sur votre Drive.')
print('=' * 75)
